# Khám phá dữ liệu — SaaS Revenue Analytics

Notebook này đọc thẳng từ `data/processed/saas.db` để kiểm tra lại các con số
trong `docs/findings.md`. Mọi truy vấn đều là SQL thuần, giống hệt nội dung
trong `sql/03_analysis/`.

Chạy `python run_all.py` trước để có database.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "processed" / "saas.db"
conn = sqlite3.connect(DB)

print(f"Database: {DB}")
print(f"Tồn tại : {DB.exists()}")
print(f"Dung lượng: {DB.stat().st_size / 1e6:.1f} MB")

Database: E:\projects\saas-revenue-analytics\data\processed\saas.db
Tồn tại : True
Dung lượng: 10.3 MB


## 1. Quy mô dữ liệu

Kiểm tra số dòng từng bảng để chắc chắn pipeline đã nạp đủ.

In [2]:
rows = []
for table in ["stg_accounts", "stg_subscription_events", "stg_invoices",
              "stg_monthly_usage", "stg_support_tickets",
              "dim_account", "fact_subscription_events", "fact_invoices",
              "fact_monthly_usage", "fact_support_tickets"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    rows.append({"bảng": table, "số dòng": f"{n:,}"})

print(pd.DataFrame(rows).to_string(index=False))

                    bảng số dòng
            stg_accounts   1,397
 stg_subscription_events   2,832
            stg_invoices  15,897
       stg_monthly_usage  15,897
     stg_support_tickets   8,476
             dim_account   1,397
fact_subscription_events   2,832
           fact_invoices  15,897
      fact_monthly_usage  15,897
    fact_support_tickets   8,476


## 2. MRR theo thời gian

MRR không được lưu sẵn. Nó luôn được tính lại bằng tổng `mrr_delta` của mọi
sự kiện tính đến thời điểm đo — một nguồn sự thật duy nhất.

In [3]:
mrr = pd.read_sql_query(
    """
    SELECT month_label, ROUND(mrr_start, 0) AS mrr_start, ROUND(mrr_end, 0) AS mrr_end,
           new_accounts, churned_accounts
    FROM mart_mrr_monthly
    ORDER BY month
    """,
    conn,
)
print("Ba tháng đầu:")
print(mrr.head(3).to_string(index=False))
print()
print("Ba tháng cuối:")
print(mrr.tail(3).to_string(index=False))
print()
print(f"MRR tăng {mrr['mrr_end'].iloc[-1] / mrr['mrr_end'].iloc[0]:.1f} lần sau 36 tháng")

Ba tháng đầu:
month_label  mrr_start  mrr_end  new_accounts  churned_accounts
    2023-01        0.0   4325.0            16                 3
    2023-02     4325.0  13676.0            16                 1
    2023-03    13676.0  25556.0            15                 3

Ba tháng cuối:
month_label  mrr_start  mrr_end  new_accounts  churned_accounts
    2025-10   626796.0 666486.0            66                19
    2025-11   666486.0 695393.0            56                16
    2025-12   695393.0 729625.0            42                22

MRR tăng 168.7 lần sau 36 tháng


## 3. Cầu nối MRR — tăng trưởng đến từ đâu?

Đây là câu hỏi trung tâm của dự án. Bốn thành phần: khách mới, mở rộng,
thu hẹp, mất do huỷ.

In [4]:
print(pd.read_sql_query(
    "SELECT component, ROUND(mrr, 0) AS mrr, pct_of_gross, direction "
    "FROM mart_growth_decomposition ORDER BY mrr DESC",
    conn,
).to_string(index=False))

new_mrr = conn.execute(
    "SELECT mrr FROM mart_growth_decomposition WHERE component = 'new'"
).fetchone()[0]
exp_mrr = conn.execute(
    "SELECT mrr FROM mart_growth_decomposition WHERE component = 'expansion'"
).fetchone()[0]
print()
print(f"Tỷ trọng tăng trưởng đến từ khách cũ: {exp_mrr / (new_mrr + exp_mrr) * 100:.1f}%")
print("Phần còn lại phải mua bằng ngân sách marketing.")

  component       mrr  pct_of_gross direction
        new  689721.0         63.06  increase
  expansion  222008.0         20.30  increase
contraction   -4923.0          0.45  decrease
      churn -177181.0         16.20  decrease

Tỷ trọng tăng trưởng đến từ khách cũ: 24.4%
Phần còn lại phải mua bằng ngân sách marketing.


## 4. Cohort retention

Mỗi hàng là một lứa khách đăng ký cùng tháng. Cột là số tháng sống sót.
Mốc 0 luôn 100% theo định nghĩa.

In [5]:
cohort = pd.read_sql_query(
    """
    SELECT cohort_month, month_index, accounts_start, accounts_retained, retention_pct
    FROM mart_cohort_retention
    WHERE cohort_month BETWEEN '2023-01-01' AND '2023-12-01'
      AND month_index IN (0, 3, 6, 12)
    """,
    conn,
)
pivot = cohort.pivot(index="cohort_month", columns="month_index",
                     values="retention_pct")
pivot.columns = [f"tháng {c}" for c in pivot.columns]
print(pivot.to_string())
print()
print("Dao động rất lớn vì mỗi cohort chỉ có 15–29 tài khoản.")
print("Chênh 1 khách = chênh 3–7 điểm phần trăm.")

              tháng 0  tháng 3  tháng 6  tháng 12
cohort_month                                     
2023-01-01      100.0    75.00    75.00     50.00
2023-02-01      100.0    87.50    75.00     75.00
2023-03-01      100.0    93.33    80.00     66.67
2023-04-01      100.0    89.47    89.47     78.95
2023-05-01      100.0    88.89    77.78     66.67
2023-06-01      100.0    95.24    85.71     76.19
2023-07-01      100.0    95.24    95.24     90.48
2023-08-01      100.0    84.62    76.92     57.69
2023-09-01      100.0    92.59    81.48     66.67
2023-10-01      100.0    79.31    75.86     68.97
2023-11-01      100.0    92.31    84.62     69.23
2023-12-01      100.0    85.71    82.14     71.43

Dao động rất lớn vì mỗi cohort chỉ có 15–29 tài khoản.
Chênh 1 khách = chênh 3–7 điểm phần trăm.


## 5. NRR và GRR

NRR trộn giữ chân với bán thêm. GRR mới là thước đo giữ chân thuần.
Luôn đọc cặp này cùng nhau.

In [6]:
summary = pd.read_sql_query("SELECT * FROM mart_retention_summary", conn)
for col in summary.columns:
    print(f"{col:<24} {summary[col].iloc[0]}")
print()
print("Khoảng cách NRR − GRR chính là toàn bộ giá trị của hoạt động mở rộng.")

avg_nrr_monthly          100.24166666666667
avg_grr_monthly          98.0225
nrr_annualized_pct       102.94
grr_annualized_pct       78.69
cohorts                  24
nrr_cohort_12m_pct       103.67
grr_cohort_12m_pct       76.01

Khoảng cách NRR − GRR chính là toàn bộ giá trị của hoạt động mở rộng.


## 6. Churn theo thâm niên

Nếu chỉ được chọn một nơi để can thiệp, đây là nơi đó.

In [7]:
print(pd.read_sql_query(
    "SELECT tenure_band, account_months, churned_accounts, monthly_churn_pct "
    "FROM mart_churn_by_tenure ORDER BY tenure_band",
    conn,
).to_string(index=False))

first = conn.execute(
    "SELECT monthly_churn_pct FROM mart_churn_by_tenure "
    "WHERE tenure_band = '1_0_2m'"
).fetchone()[0]
last = conn.execute(
    "SELECT monthly_churn_pct FROM mart_churn_by_tenure "
    "WHERE tenure_band = '5_24m_plus'"
).fetchone()[0]
print()
print(f"Nhóm 0–2 tháng rời bỏ gấp {first / last:.1f} lần nhóm 24+ tháng.")

tenure_band  account_months  churned_accounts  monthly_churn_pct
     1_0_2m            4051               161               3.97
     2_3_5m            3509                89               2.54
    3_6_11m            5581                95               1.70
   4_12_23m            6091                61               1.00
 5_24m_plus            1511                13               0.86

Nhóm 0–2 tháng rời bỏ gấp 4.6 lần nhóm 24+ tháng.


## 7. Mức sử dụng và churn

Tín hiệu sớm mạnh nhất — có trước khi khách gọi điện huỷ.

In [8]:
print(pd.read_sql_query(
    "SELECT usage_band, account_months, churn_next_month, churn_rate_pct, "
    "       avg_active_days "
    "FROM mart_usage_vs_churn ORDER BY churn_rate_pct DESC",
    conn,
).to_string(index=False))

hi = conn.execute(
    "SELECT churn_rate_pct FROM mart_usage_vs_churn WHERE usage_band = 'dormant'"
).fetchone()[0]
lo = conn.execute(
    "SELECT churn_rate_pct FROM mart_usage_vs_churn WHERE usage_band = 'power'"
).fetchone()[0]
print()
print(f"Chênh lệch giữa hai cực: {hi / lo:.1f} lần.")

usage_band  account_months  churn_next_month  churn_rate_pct  avg_active_days
   dormant            1583                55            3.47              2.4
       low            8359               210            2.51              9.9
    steady            5585                88            1.58             18.3
     power             370                 3            0.81             26.2

Chênh lệch giữa hai cực: 4.3 lần.


## 8. Một bẫy phương pháp: thiên lệch sống sót

Cách làm trực quan — đếm số ticket mỗi tài khoản rồi so tỷ lệ huỷ — cho kết quả
**ngược hoàn toàn**. Cùng một dữ liệu, hai cách đếm, hai kết luận trái ngược.

In [9]:
wrong = pd.read_sql_query(
    """
    WITH base AS (
        SELECT a.account_id, COALESCE(t.n, 0) AS n_tickets,
               CASE WHEN c.account_id IS NULL THEN 0 ELSE 1 END AS churned
        FROM dim_account a
        LEFT JOIN (SELECT account_id, COUNT(*) AS n FROM fact_support_tickets
                   GROUP BY account_id) t ON t.account_id = a.account_id
        LEFT JOIN (SELECT DISTINCT account_id FROM fact_subscription_events
                   WHERE is_churn = 1) c ON c.account_id = a.account_id
    )
    SELECT CASE WHEN n_tickets = 0 THEN '0 ticket'
                WHEN n_tickets <= 2 THEN '1-2 ticket'
                WHEN n_tickets <= 5 THEN '3-5 ticket'
                ELSE '6+ ticket' END AS nhom,
           COUNT(*) AS accounts,
           ROUND(100.0 * SUM(churned) / COUNT(*), 2) AS churn_pct
    FROM base GROUP BY 1 ORDER BY MIN(n_tickets)
    """,
    conn,
)
print("CÁCH SAI — đếm theo tài khoản:")
print(wrong.to_string(index=False))
print("→ Nhìn như thể mở nhiều ticket thì ít huỷ. Vô lý.")
print("→ Nguyên nhân: khách rời sớm không có thời gian mở ticket.")
print()

right = pd.read_sql_query(
    "SELECT satisfaction_band, tickets, churn_within_3m, churn_rate_pct "
    "FROM mart_satisfaction_band ORDER BY satisfaction_band",
    conn,
)
print("CÁCH ĐÚNG — đếm theo ticket, cửa sổ 3 tháng:")
print(right.to_string(index=False))
print("→ Điểm hài lòng thấp đi kèm tỷ lệ huỷ cao hơn rõ rệt.")

CÁCH SAI — đếm theo tài khoản:
      nhom  accounts  churn_pct
  0 ticket       176      49.43
1-2 ticket       373      42.90
3-5 ticket       324      27.78
 6+ ticket       524      15.65
→ Nhìn như thể mở nhiều ticket thì ít huỷ. Vô lý.
→ Nguyên nhân: khách rời sớm không có thời gian mở ticket.

CÁCH ĐÚNG — đếm theo ticket, cửa sổ 3 tháng:
satisfaction_band  tickets  churn_within_3m  churn_rate_pct
            1_1_2      529               44            8.32
              2_3     1914              136            7.11
              3_4     4339              282            6.50
              4_5     1694              120            7.08
→ Điểm hài lòng thấp đi kèm tỷ lệ huỷ cao hơn rõ rệt.


## 9. Bảng rủi ro tài khoản

Đầu ra dùng được ngay trong vận hành — không cần thêm phân tích nào.

In [10]:
print(pd.read_sql_query(
    "SELECT risk_flag, accounts, pct_accounts, ROUND(active_mrr, 0) AS active_mrr, "
    "       pct_active_mrr FROM mart_risk_summary ORDER BY accounts DESC",
    conn,
).to_string(index=False))
print()
print("Top 5 tài khoản rủi ro cao nhất (theo MRR):")
print(pd.read_sql_query(
    "SELECT account_id, account_name, tier, ROUND(mrr_last, 0) AS mrr, "
    "       last_active_days, last_seat_utilization, failed_invoices "
    "FROM mart_account_risk WHERE risk_flag = 'at_risk' "
    "ORDER BY mrr_last DESC LIMIT 5",
    conn,
).to_string(index=False))

risk_flag  accounts  pct_accounts  active_mrr  pct_active_mrr
    watch       518         37.08    331876.0           45.49
  churned       419         29.99         0.0            0.00
  healthy       279         19.97    317147.0           43.47
  at_risk       181         12.96     80602.0           11.05

Top 5 tài khoản rủi ro cao nhất (theo MRR):
account_id      account_name       tier    mrr  last_active_days  last_seat_utilization  failed_invoices
 ACC-00953      Vertex Group   business 3207.0                 1                 0.3646                0
 ACC-00018    Crown Networks   business 3009.0                 2                 0.4059                1
 ACC-01014 Solstice Partners enterprise 2960.0                 1                 0.3538                0
 ACC-00167      Ash Partners enterprise 2564.0                14                 0.6429                2
 ACC-00916   Iron Industries enterprise 1838.0                13                 0.4430                0


## 10. Kết luận

Ba điều notebook này xác nhận lại từ dữ liệu gốc:

1. **Tăng trưởng đến từ khách mới.** 63% MRR tăng thêm là khách mới; phần mở
   rộng chỉ bù được một phần tổn thất do huỷ (GRR 76%).
2. **Ba tháng đầu quyết định vòng đời khách.** Xác suất huỷ giảm từ 3,97%/tháng
   xuống 0,86% — chênh 4,6 lần.
3. **Mức sử dụng là tín hiệu sớm tốt nhất.** Chênh 4,3 lần giữa nhóm không dùng
   và nhóm tích cực.

Chi tiết đầy đủ và khuyến nghị: `docs/findings.md`.